# 검증 시점 근거로 후보 선택 논리 추적하기

## 이번 질문

슬라이드의 모델 선택은 PhysioNet에서 GridSearch를 돌리는 것이 아닙니다. 봉인된 `test`로 후보를 고르는 것도 아닙니다. 세 후보는 이미 `profiles.yaml`에 고정되어 있고, `development-benchmark.json`은 검증 시점 비교입니다. 이 노트북은 학습하지 않고, MLflow에 쓰지 않고, `test.csv`를 열지 않습니다. 선언과 검증 숫자를 한 표로 읽어 선택 논리만 추적합니다.


## 먼저 예상

클래스 가중과 예측 임계값이 다른 세 프로필 가운데, 검증 재현율과 미탐이 어떻게 갈릴지 한 문장으로 적습니다. 이 표의 숫자가 공식 승인인지 아닌지도 예상합니다.

## 실행과 관측

입력은 모델 프로필 YAML과 검증 시점 비교 JSON뿐입니다. 모델을 맞추거나 공식 평가 JSON을 열지 않습니다.

### 1. 읽을 경로

저장소 루트에서 두 파일을 엽니다. 학습, MLflow 기록, 봉인 test는 이 활동의 범위가 아닙니다.


In [ ]:
from pathlib import Path

import json
import pandas as pd
import yaml

# 1. 노트북을 하위 폴더에서 열어도 저장소 루트를 찾는다.
ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir()
)

# 2. 이미 있는 선언과 검증 시점 비교만 연다. 학습하지 않는다.
PROFILES_PATH = ROOT / "configs/model-v2/profiles.yaml"
BENCHMARK_PATH = (
    ROOT / "docs/evidence/model-v2/development-benchmark.json"
)
profiles = yaml.safe_load(PROFILES_PATH.read_text(encoding="utf-8"))
evidence = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))

pd.DataFrame(
    {
        "역할": ["모델 선언", "검증 시점 비교"],
        "경로": [
            str(PROFILES_PATH.relative_to(ROOT)),
            str(BENCHMARK_PATH.relative_to(ROOT)),
        ],
        "평가 역할": ["해당 없음", evidence["evaluation_role"]],
    }
)


### 2. 이미 고른 세 프로필

종류, 클래스 가중, 임계값은 코드 기본값이 아니라 YAML 선언입니다. 이 노트북에서 값을 바꾸거나 다시 고르지 않습니다.


In [ ]:
# 1. 세 프로필의 종류, 클래스 가중, 임계값만 표로 읽는다.
# 2. GridSearch를 돌리거나 임계값을 다시 고르지 않는다.
rows = []
for profile in profiles["profiles"]:
    params = profile.get("params", {})
    rows.append(
        {
            "profile": profile["name"],
            "kind": profile["kind"],
            "class_weight": params.get("class_weight"),
            "threshold": profile["threshold"],
        }
    )
declared = pd.DataFrame(rows).set_index("profile")
declared


### 3. 검증 시점 지표와 교차검증 재현율

아래 숫자는 개발 `valid` 평가입니다. 공식 봉인 test가 아닙니다. 교차검증 재현율 평균이 있으면 함께 붙입니다.


In [ ]:
# 1. 검증 JSON의 재현율, 정밀도, 미탐을 읽는다.
# 2. 교차검증 재현율 평균이 있으면 같은 표에 붙인다.
rows = []
for item in evidence["profiles"]:
    metrics = item["metrics"]
    cv_recall = (item.get("cross_validation") or {}).get("recall") or {}
    rows.append(
        {
            "profile": item["profile"],
            "valid_recall": metrics["recall"],
            "valid_precision": metrics["precision"],
            "valid_false_negative": metrics["false_negative"],
            "cv_recall_mean": cv_recall.get("mean"),
        }
    )
valid_metrics = pd.DataFrame(rows).set_index("profile")

# 3. 선언과 검증 숫자를 한 표로 붙인다. 공식 승인 열은 넣지 않는다.
comparison = declared.join(valid_metrics)
comparison.round(4)


### 4. 검증 숫자만으로 말할 수 있는 것과 없는 것

재현율과 미탐이 갈리는 방향은 이 표에서 보입니다. 그래도 이 숫자는 공식 승인 결과가 아닙니다.


In [ ]:
# 1. 검증 재현율이 높은 쪽과 미탐이 적은 쪽만 표시한다.
# 2. HOLD/APPROVE를 여기서 매기지 않는다. 그 판단은 다음 노트북의 봉인 JSON이다.
reading = pd.DataFrame(
    {
        "valid_recall": comparison["valid_recall"],
        "valid_precision": comparison["valid_precision"],
        "valid_false_negative": comparison["valid_false_negative"],
        "highest_valid_recall": comparison["valid_recall"].eq(
            comparison["valid_recall"].max()
        ),
        "lowest_valid_fn": comparison["valid_false_negative"].eq(
            comparison["valid_false_negative"].min()
        ),
    }
)
reading.round(4)


## 해석과 기록

이 표의 검증 재현율, 정밀도, 미탐은 공식 승인이 아닙니다. 후보는 이미 YAML에서 고른 뒤 검증 시점에서 비교한 근거입니다. 다음 노트북은 봉인된 공식 평가 JSON을 읽고 Candidate A 보류와 Candidate B 승인을 확인합니다.

## 결과 점검


In [ ]:
accessed_roles = set(evidence["accessed_roles"])
assert evidence["evaluation_role"] == "valid"
assert accessed_roles == {"train", "valid"}
assert list(declared.index) == ["baseline", "candidate-a", "candidate-b"]
assert {"kind", "class_weight", "threshold"} <= set(declared.columns)
assert {
    "valid_recall",
    "valid_precision",
    "valid_false_negative",
    "cv_recall_mean",
} <= set(comparison.columns)
assert comparison["cv_recall_mean"].notna().all()
print(
    "검증 시점 비교를 읽었습니다. "
    "이 숫자는 공식 승인이 아니며, 다음 노트북은 봉인 test JSON입니다."
)


## 다음 확인

다음 노트북 `01_compare_model_evidence.ipynb`는 봉인된 `canonical-benchmark.json`만 읽습니다. 그 파일은 공식 평가용 `test`를 한 번 연 결과입니다. 이 노트북의 검증 숫자를 공식 승인 칸에 옮기지 않습니다.
